# Flight Delay and Cancellation Analysis (2019-2023)

## 1. Problem Formulation (Phase 1)
O dataset contém informações sobre voos comerciais nos EUA. O problema central que este projeto aborda é a previsão de atrasos na chegada de voos, garantindo que apenas utilizamos informações disponíveis antes da partida para evitar fuga de dados (*data leakage*).

## 2. Goals and Objectives
A análise tem os seguintes objetivos principais:
* **Regressão:** Prever a duração do atraso na chegada (`ARR_DELAY`) em minutos.
* **Classificação:** Reformular o problema para prever 3 classes de atraso (On-time: < 15 min, Short delay: 15-30 min, Long delay: > 30 min).
* **Clustering:** Identificar padrões de desempenho operacional em aeroportos e companhias aéreas.
* **Testes de Hipóteses:** Validar estatisticamente hipóteses (ex: atrasos sistemáticos de certas companhias ou impacto do clima).


## Setup of environment

Initial initializations of data structures needed to perform analysis as well as source code.

In [ ]:
import pandas as pd
import numpy as np

# Auto-reload for development
%load_ext autoreload
%autoreload 2

# Import your custom module
from PythonCode.DataPreProcessor import FlightDataProcessor

#Initialize data structures that are needed
loader = FlightDataProcessor.FlightDataLoader(DataPath='../../DataSets/flights_sample_3m.csv')
df = loader.load_data()
cleaner = FlightDataProcessor.FlightDataCleaner(df)


## Clean code

In [ ]:
# 2. Filtrar voos Cancelados e Desviados
# Voos cancelados ou desviados não têm valores de atraso na chegada (ARR_DELAY) com significado
print(f"Tamanho original do dataset: {df.shape}")
cleaner.remove_cancelled_diverted()
print(f"Tamanho após remover cancelados e desviados: {df.shape}")

# 3. Prevenção de Data Leakage (Fuga de Informação)
# Remover colunas que contêm informação pós-partida ou que derivam diretamente da variável alvo [cite: 182]
cleaner.remove_data_leak_cols()

#4. Tratamento de Missing Values ---
print("\n--- A tratar Missing Values ---")
cleaner.fill_missing(mean_strategy='median')

print(f"Tamanho do dataset após remover NaNs: {df.shape}")

cleaner.handle_missing_values(method='drop')


In [1]:
# Importação de bibliotecas base (baseado no pythonTP2.py)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# Ignorar avisos futuros para manter o output limpo
warnings.simplefilter(action='ignore', category=FutureWarning)

# 1. Carregar o dataset
print("A carregar o dataset...")
df = pd.read_csv('../../DataSet/flights_sample_3m.csv')




# 4. Análise Exploratória Inicial (EDA)
print("\n--- Informação Básica do Dataset ---")
df.info()

print("\n--- Primeiras linhas do Dataset ---")
print(df.head())

print("\n--- Verificação de Valores Nulos (Missing Values) ---")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])



# --- TAREFA 9: Tratamento de Outliers (Método IQR) ---
print("\n--- A tratar Outliers nas variáveis contínuas preditoras ---")
# Aplicamos o método IQR (visto no pythonTP2.py) apenas a features contínuas.
# NÃO aplicamos à variável alvo (ARR_DELAY) porque queremos prever os grandes atrasos!
cols_for_outliers = ['DISTANCE', 'CRS_ELAPSED_TIME']

for col in cols_for_outliers:
    # Calcular o IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    # Definir limites
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Substituir outliers por NaN e imputar com a média (conforme guião pythonTP2.py)
    df[col] = np.where(
        (df[col] < lower_bound) | (df[col] > upper_bound),
        np.nan,
        df[col]
    )
    # Imputar os NaNs gerados com a média da coluna
    df[col] = df[col].fillna(df[col].mean())

print("Tratamento de Outliers concluído.")

# Verificação final para garantir que não há mais nulos
print("\nVerificação final de NaNs:")
print(df[['ARR_DELAY', 'DISTANCE', 'CRS_ELAPSED_TIME']].isnull().sum())

A carregar o dataset...
Tamanho original do dataset: (3000000, 32)
Tamanho após remover cancelados e desviados: (2913804, 32)

--- Informação Básica do Dataset ---
<class 'pandas.core.frame.DataFrame'>
Index: 2913804 entries, 0 to 2999999
Data columns (total 15 columns):
 #   Column            Dtype  
---  ------            -----  
 0   FL_DATE           object 
 1   AIRLINE           object 
 2   AIRLINE_DOT       object 
 3   AIRLINE_CODE      object 
 4   DOT_CODE          int64  
 5   FL_NUMBER         int64  
 6   ORIGIN            object 
 7   ORIGIN_CITY       object 
 8   DEST              object 
 9   DEST_CITY         object 
 10  CRS_DEP_TIME      int64  
 11  CRS_ARR_TIME      int64  
 12  ARR_DELAY         float64
 13  CRS_ELAPSED_TIME  float64
 14  DISTANCE          float64
dtypes: float64(3), int64(4), object(8)
memory usage: 355.7+ MB

--- Primeiras linhas do Dataset ---
      FL_DATE                AIRLINE                AIRLINE_DOT AIRLINE_CODE  \
0  2019-01-09  Unite

## Pre-processing
1. Deleting record
2. Imputation
3. Filtering
1. Describe the dataset, its source, and any preprocessing steps taken.
2. Include data cleansing and normalization/standardization.

## Exploratory Data Analysis (EDA)
1. Conduct descriptive statistics and visualizations to understand the data.
2. Use standard statistical methods (such as histograms) to identify
patterns, outliers, and correlations.
3. Use dimension reduction methods to identify patterns in the data - PCA and UMAP
4. Discuss any initial insights gained from EDA.
## Hypothesis Testing
1. Formulate null and alternative hypotheses.
2. Choose appropriate statistical tests.
3. Perform hypothesis testing and interpret the results